# SLEAP Posture Tracking and Registration Pipeline - Freely Swimming

This notebook processes video recordings with SLEAP-based posture tracking and image registration for freely swimming experiments.

## Pipeline Overview:
1. **Load experimental data**: Read HDF5 files containing motor positions and timestamps
2. **Load SLEAP labels**: Read posture tracking data from SLEAP analysis files
3. **Spline tracking**: Extract body spline from SLEAP keypoints
4. **Image registration**: Align images using head region for consistent orientation
5. **Post-processing**: Transform coordinates to laboratory reference frame
6. **Save results**: Export processed data to HDF5 format

## Configuration
**IMPORTANT**: Update the paths in the "Set path" section below to match your data directory structure.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'    

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

import time
import datetime
from time import sleep
from tqdm.notebook import tqdm

from numba import njit

import imageio.v2 as imageio
from PIL import Image
import re

from matplotlib import animation
from matplotlib.patches import Rectangle, Circle
import matplotlib.patches as patches

from ipywidgets import interact,IntSlider

from skimage import data, color
from skimage.transform import rescale, resize, downscale_local_mean
from skimage import io, draw
from skimage.draw import disk
from skimage.registration import phase_cross_correlation
from skimage.io import imsave

import math
from scipy.ndimage import rotate
from scipy.interpolate import interp1d

import h5py

In [ ]:
from utilities.saving import saveList, loadList
from utilities.reading_movie_manage_batch import readingMovie,sorted_alphanumeric,read_mp4_video,process_batch
from utilities.reading_data_from_exp import h5tree_view, extract_time_from_date
from utilities.reading_data_from_sleap import read_sleap_labels, quantities_from_sleap
from utilities.reading_movie_manage_batch import process_batch
from utilities.plotting_postprocessing import fig_traj_allRuns, plot_test_spline, plot_random_frames_test_spline_avg, plot_test_final, plot_random_frames_test_final
from utilities.tracking_spline import spline_tracking, discared_frames_based_on_scores, spline_tracking_new
from utilities.tracking_registration import def_reg_image, reg_angle_single_frame, cropping_img, calculate_angle, preparation4registration,  registration_process
from utilities.def_com_spline_orient_ import undo_crop_and_rotation, calculate_orientation,rotate_2darray,create_nan_array_like,convert_pix2mm, transform_points_after_reg
from utilities.analysis_spline import moving_rms, spline_transformed, reconstruct_segments, extract_and_reconstruct, calculate_spline_length, PCA_spline, spline_length_mm


In [ ]:
plot_debug=True

# Configuration: Set Data Paths

**IMPORTANT**: Configure the paths below to point to your data directory.

The expected directory structure is:
```
postprocessed_fs/  (freely swimming experiments)
└── [experiment_day]/  (e.g., fs_14dpf_27Mar24)
    └── fish_[number]/
        ├── converted_movie/      # MP4 video files
        ├── labels_sleap/        # SLEAP analysis HDF5 files
        ├── experiment_runs/     # Experimental HDF5 files (run*.h5)
        ├── postprocessed_runs/  # Output: processed data
        └── plots/               # Output: visualization plots
```

**Note**: Modify `base_path` below.

In [ ]:
# ============================================================================
# CONFIGURATION: Experimental paths
# ============================================================================

from pathlib import Path
NOTEBOOK_DIR = Path.cwd()

DATA_DIR = None
for parent in [NOTEBOOK_DIR] + list(NOTEBOOK_DIR.parents):
    candidate = parent / "Tracking_camera" / "data" / "freely_swimming"
    if candidate.exists():
        DATA_DIR = candidate
        REPO_ROOT = parent
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate 'Tracking_camera/data/freely_swimming'.\n\n"
        "Searched upward from:\n"
        f"{NOTEBOOK_DIR}\n\n"
        "Expected somewhere above:\n"
        "Tracking_camera/data/freely_swimming/"
    )

print(f"Using data directory:\n  {DATA_DIR}")
print(f"Detected repo root:\n  {REPO_ROOT}")

print("\nAvailable experiment folders:")
for p in sorted(DATA_DIR.iterdir()):
    if p.is_dir() and not p.name.startswith("."):
        print(f"  - {p.name}")

In [ ]:
# ============================================================================
# CONFIGURATION: Set experiment day and fish number
# ============================================================================
# Update these variables to match your experiment
experiment_day = 'fs_27dpf_12Jun24'  # Change this to your experiment day folder name

dir_experiment_day = os.path.join(DATA_DIR, experiment_day)
if not os.path.exists(dir_experiment_day):
    raise FileNotFoundError(f'ERROR: Experiment folder does not exist: {dir_experiment_day}\n'
                           f'Available folders in {DATA_DIR}:\n'
                           f'{[f for f in os.listdir(DATA_DIR) if not f.startswith(".")]}')
else:
    print(f"✓ Experiment day folder found: {dir_experiment_day}")
    print("\nContents:")
    for file in os.listdir(dir_experiment_day):
        if not file.startswith('.'):  # Skip hidden files
            print(f"  - {file}")

# Path for calibration data (background image, etc.)
path_calib = dir_experiment_day

In [ ]:
# Set fish number (update this to process a different fish)
fish_number = '1'  # Change this to '1', '2', '3', '4', '5', etc.

In [ ]:
# Build path to fish-specific directory
# Note: For freely swimming experiments, there is no experimental condition subfolder
dir_experiment_fish = os.path.join(dir_experiment_day, 'fish_' + str(fish_number))
if not os.path.exists(dir_experiment_fish):
    raise FileNotFoundError(f'ERROR: Fish folder does not exist: {dir_experiment_fish}\n'
                           f'Available fish folders:\n'
                           f'{[f for f in os.listdir(dir_experiment_day) if f.startswith("fish_")]}')
else:
    print(f'✓ Fish folder found: {dir_experiment_fish}')

In [ ]:
# ============================================================================
# Define all data paths
# ============================================================================
path_converted_movie = os.path.join(dir_experiment_fish, 'converted_movie')
path_labels_sleap = os.path.join(dir_experiment_fish, 'labels_sleap')
path_experiment_runs = os.path.join(dir_experiment_fish, 'experiment_runs')
path_postprocessed_runs = os.path.join(dir_experiment_fish, 'postprocessed_runs')
path_plots = os.path.join(dir_experiment_fish, 'plots')

# Validate that required input directories exist
required_dirs = {
    'converted_movie': path_converted_movie,
    'labels_sleap': path_labels_sleap,
    'experiment_runs': path_experiment_runs
}

print("Data paths:")
print(f"  converted_movie:     {path_converted_movie}")
print(f"  labels_sleap:        {path_labels_sleap}")
print(f"  experiment_runs:     {path_experiment_runs}")
print(f"  postprocessed_runs:   {path_postprocessed_runs}")
print(f"  plots:               {path_plots}")

print("\nValidating required directories:")
for name, dir_path in required_dirs.items():
    if os.path.exists(dir_path):
        file_count = len([f for f in os.listdir(dir_path) if not f.startswith('.')])
        print(f"  ✓ {name}: {dir_path} ({file_count} files)")
    else:
        print(f"  ✗ {name}: {dir_path} (NOT FOUND)")
        print(f"    Please ensure this directory exists and contains the required data files.")

In [ ]:
# Count the number of experimental runs available
n_run_tot = 0
run_files = []
for file in os.listdir(path_experiment_runs):
    if file.startswith('run') and file.endswith('.h5'):
        n_run_tot += 1
        run_files.append(file)

print(f'Number of total runs: {n_run_tot}')
if n_run_tot > 0:
    print("Available run files:")
    for f in sorted(run_files):
        print(f"  - {f}")
else:
    print("WARNING: No run files found. Please ensure experiment_runs directory contains run*.h5 files.")

# Quick Overview: Visualize Trajectories

This section loads trajectory data from all runs and creates a quick visualization to verify data integrity.

In [ ]:
tot_traj=[[] for i in range(n_run_tot)]
time_traj=[[] for i in range(n_run_tot)]
category_names=[[] for i in range(n_run_tot)]
date_f_run=[[] for i in range(n_run_tot)]
date_i_run=[[] for i in range(n_run_tot)]
date_in_arena=[[] for i in range(n_run_tot)]

for i,file in enumerate(os.listdir(path_experiment_runs)):
    #print(f'{i}:, {file}')
    h5_experiment_run=os.path.join(path_experiment_runs,file)
    hdf5_file = h5py.File(h5_experiment_run, "r") #h5tree_view(hdf5_file)
    tot_traj[i]=hdf5_file['data/motorpositions'][:]
    date_f_run[i]=hdf5_file['data/date_f_run'][:]
    date_i_run[i]=hdf5_file['data/date_i_run'][:]
    fr_camera=hdf5_file['data/cameratimes'][:] 
    aux=hdf5_file['data/motortimes'][:]  
    time_traj[i]=round((aux[-1]-aux[0])/60,2)
    category_names[i]='run'+str(i+1)

In [ ]:
fps=int(1/np.mean(np.diff(fr_camera)))
print(f'fps recording {fps}')

In [ ]:
time_runs = [ extract_time_from_date(date_i_run[i][0]) for i in range(n_run_tot)]
time_runs

In [ ]:
duration_runs = {'tot. time\n'+str(round(np.sum(time_traj)))+'min': time_traj}

title=dir_experiment_day+'_fish'+ str(fish_number)
[fig,ax,category_colors]=fig_traj_allRuns(duration_runs,time_runs,category_names,title)
#ax[1].imshow(new_im_array[:,::-1],origin='lower',alpha=0.3,extent=[x_center_motor-im_pos[0]-im_size[0]/2,x_center_motor-im_pos[0]+im_size[0]/2,y_center_motor-im_pos[1]-im_size[1]/2,y_center_motor-im_pos[1]+im_size[1]/2])
for i in range(n_run_tot):
    ax[1].plot(tot_traj[i][:,0],tot_traj[i][:,1],c=category_colors[i],alpha=1)
    ax[1].set_xlabel('X (mm)')
    ax[1].set_ylabel('Y (mm)')
    #ax[1].axis('equal')
    ax[1].set_xlim([0,400])
    ax[1].set_ylim([0,400])
plt.show()

# START

# Manual run by run

In [ ]:
run_n='run1'

In [ ]:
for i,file in enumerate(os.listdir(path_experiment_runs)):
    if run_n in file: #loop on the different runs
        #print(file)
        h5_experiment_run=os.path.join(path_experiment_runs,file)
        hdf5_file = h5py.File(h5_experiment_run, "r") 
        #h5tree_view(hdf5_file)   
        fps=hdf5_file['exp/fps'][:][0]
        mm_per_pix=hdf5_file['exp/mm_per_pix'][:][0]      
        age=hdf5_file['fish_id/age'][:]
        cameratimes=hdf5_file['data/cameratimes'][:]
        cameratimestamp=hdf5_file['data/cameratimestamp'][:]
        pos_fish_yolo =hdf5_file['data/pos_fish_yolo'][:]
        motorpositions=hdf5_file['data/motorpositions'][:]
        motortimes=hdf5_file['data/motortimes'][:]
        trackingtimes=hdf5_file['data/trackingtimes'][:]
        date_f_run=hdf5_file['data/date_f_run'][:]
        date_i_run=hdf5_file['data/date_i_run'][:]
        date_in_arena=hdf5_file['fish_id/date_in_arena'][:]
        hdf5_file.close()
#print(f'{date_in_arena}\n{date_i_run}\n{date_f_run}')

In [ ]:
file_pairs = []

# Collect video files and their base names
video_files = {file.replace('.mp4', ''): os.path.join(path_converted_movie, file) for file in os.listdir(path_converted_movie) if run_n+'_' in file}

for file in os.listdir(path_labels_sleap):
    if run_n in file:
        # Regular expression pattern to extract the part we need from the filename
        pattern = r'converted_run\d+_movie\d+'
        
        # Search for the pattern in the filename
        match = re.search(pattern, file)
        
        # Check if a match is found and extract the matched string
        if match:
            extracted_key = match.group(0)
          #  print(f"Extracted key: {extracted_key}")
            
            # Access the value using the extracted key
            if extracted_key in video_files:
                video_path = video_files[extracted_key]
                label_path = os.path.join(path_labels_sleap, file)
                file_pairs.append((video_path, label_path))
                #print(f"Key '{extracted_key}' found in the dictionary. \n file_pairs: '{file_pairs}'")
            else:
                print(f"Key '{extracted_key}' not found in the dictionary.")
        else:
            print("Pattern not found in the filename.")
            
import re

def extract_movie_index(path):
    match = re.search(r'movie(\d+)', path)
    return int(match.group(1)) if match else -1

# Sort the list of (video, label) pairs by the numeric movie index
file_pairs_sorted = sorted(file_pairs, key=lambda pair: extract_movie_index(pair[0]))
for video, label in file_pairs_sorted:
    print(f"Matched video: {video} with label: {label}")

In [ ]:
window_radius=5 #5pix
n_segments=20

r_len_rec=[]
r_head_L=[]
r_mean_head_L=[]
r_profile_L=[]
r_mean_fish_L=[]
r_orient_sleap=[]

r_discarded_frame=[]
r_discarded_frame_set=[]

r_start_crop=[]
r_end_crop=[]

r_reg_transl=[]
r_reg_angle=[]

r_spline_sy0=[]
r_com_sy0=[]
r_orient=[]

index=0

for video, label in file_pairs_sorted:
    #print(f"Matched video: {video} with label: {label}")
    print('Start reading video', video)
    [myVideo, width, height, fps, total_frames]=read_mp4_video(os.path.join(path_converted_movie,video),True)
    print(f"myVideo is loaded with shape {myVideo.shape}")
    print('Start loading sleap labels', label)
    filename=os.path.join(path_labels_sleap,label)

    [instance_scores,point_scores, head_loc, t1_loc, t2_loc, t3_loc, t4_loc, fl1_loc, fl2_loc, fr1_loc, fr2_loc]=read_sleap_labels(filename, show_details=False)
   
    if myVideo.shape[2]==instance_scores.shape[1]==head_loc.shape[0]:
        len_rec=myVideo.shape[2]
        print(f'ok! tot. frame={len_rec}')
    else:
        len_rec=instance_scores.shape[1]
        myVideo=myVideo[:,:,:len_rec]
    #from sleap    
    th_score = 1
    th_score_points = 0.1 #0.5
    [id_score, head_L, mean_head_L, profile_L, mean_fish_L, head_direction, tail_direction, head_direction_unwrap]=quantities_from_sleap(instance_scores, th_score, head_loc, t1_loc, t2_loc, t3_loc, t4_loc) 
    if not np.isnan(mean_head_L) and not np.isnan(mean_fish_L):
        #spline tracking 
        spline_avg  = spline_tracking(myVideo, profile_L, head_L, mean_fish_L, mean_fish_L, n_segments, window_radius, tail_direction, t4_loc , head_loc)
        #discard frames with too low score
        [ok_frame, discarded_frame, discarded_frame_set] = discared_frames_based_on_scores(len_rec,instance_scores, point_scores, head_loc, t1_loc, t2_loc, t3_loc, t4_loc, th_score = th_score, th_score_points = th_score_points)
        #plot1:plot_random_frames_test_spline_avg
        plot_random_frames_test_spline_avg(len_rec, myVideo, spline_avg, t1_loc, head_loc, t2_loc, t3_loc, t4_loc, discarded_frame_set,instance_scores)
        #preparation for registration: cropping+rotation
        [ head_reg, start_crop, end_crop, orient_sleap] = preparation4registration(len_rec, spline_avg, discarded_frame_set, myVideo, t1_loc, head_loc)
        #define the reference image
        fix_x, fix_y  = 22, 15
        ref_img, fix_x, fix_y = def_reg_image(len_rec, head_reg, fix_x, fix_y)
        fixed_points = np.array([(fix_x, fix_y), (fix_x-10, fix_y)])
        #go with the registration
        video_reg, reg_transl, reg_angle = registration_process(head_reg, ref_img, discarded_frame_set)  
        #transform center_of_mass and spline with registration corrections
        fixed_points_after_reg = transform_points_after_reg(fixed_points, head_reg, reg_angle, reg_transl, discarded_frame_set)
        points_sy0 = undo_crop_and_rotation(fixed_points_after_reg, orient_sleap, start_crop, end_crop)
        window_radius = 5
       # mean_fish_L = 75
        spline_sy0  = spline_tracking_new(myVideo, points_sy0, mean_fish_L, n_segments, window_radius)
        com_sy0 = [spline[5] if spline is not None and len(spline) > 5 else np.array([np.nan, np.nan]) for spline in spline_sy0]
        #orientation:definition+plot
        orient=calculate_orientation(orient_sleap, reg_angle)
        orient=orient+180
        #final plot
        plot_random_frames_test_final(len_rec, myVideo, spline_sy0, com_sy0, discarded_frame_set, instance_scores, orient)    
    else:
        orient_sleap=create_nan_array_like(orient_sleap,len_rec)
        discarded_frame=create_nan_array_like(discarded_frame,len_rec)
        start_crop=create_nan_array_like(start_crop,len_rec)
        end_crop=create_nan_array_like(end_crop,len_rec)
        reg_transl=create_nan_array_like(reg_transl,len_rec)
        reg_angle=create_nan_array_like(reg_angle,len_rec)
        spline_sy0=create_nan_array_like(spline_sy0,len_rec)
        com_sy0=create_nan_array_like(com_sy0,len_rec)
        orient=create_nan_array_like(orient,len_rec)

    r_len_rec.append(len_rec)
    r_mean_head_L.append(mean_head_L)
    r_mean_fish_L.append(mean_fish_L)
    r_orient_sleap.append(orient_sleap)

    r_discarded_frame.append(discarded_frame)

    r_start_crop.append(start_crop)
    r_end_crop.append(end_crop)

    r_reg_transl.append(reg_transl)
    r_reg_angle.append(reg_angle)
 
    r_spline_sy0.append(spline_sy0)
    r_com_sy0.append(com_sy0)
    r_orient.append(orient)

    if len_rec==len(r_orient_sleap[index])==len(r_reg_transl[index])==len(r_reg_angle[index])==len(r_spline_sy0[index])==len(r_com_sy0[index])==len(r_orient[index]):
        print('ok')
    else:
        print('ERROR: the dimentions are not consistent')
        break
    index=index+1 
img_ref=myVideo[:,:,0]
xROIcenter=img_ref.shape[0]/2
yROIcenter=img_ref.shape[1]/2
w_img=img_ref.shape[0]
h_img=img_ref.shape[1]
print(xROIcenter,yROIcenter,w_img,h_img)

In [ ]:
plot_debug = True
%matplotlib widget
if plot_debug:
    def slider_movie(frame):
        plt.close()
        plt.figure(figsize=(5,2),dpi=250)
        plt.imshow(myVideo[:,:,frame], origin='lower', cmap='gray')
        plt.plot(t1_loc[frame][0], t1_loc[frame][1], '.')
        plt.plot(head_loc[frame][0], head_loc[frame][1], '.')
        plt.plot(t2_loc[frame][0], t2_loc[frame][1], '.')
        plt.plot(t3_loc[frame][0], t3_loc[frame][1], '.')
        plt.plot(t4_loc[frame][0], t4_loc[frame][1], '.')
        
   
        if frame not in discarded_frame_set:
            #plt.plot(points_sy0[frame][:, 0],points_sy0[frame][:, 1],'rx')
            plt.plot(spline_sy0[frame][:, 0],spline_sy0[frame][:, 1],'r-')
            
            plt.plot(com_sy0[frame][0],com_sy0[frame][1],'x')
        else:
            plt.title('not tracked frame')
        
        plt.grid()
        plt.show()
    interact(slider_movie,frame=IntSlider(value=11157,min=0,max=len_rec-1,step=1))

In [ ]:
%matplotlib inline

if plot_debug:
    def slider_movie(frame):
        plt.close()
        fig, axes = plt.subplots(1, 3, figsize=(8, 3), dpi=150)  # 1 row, 2 columns

        # First subplot: registered image
        ax = axes[0]
        ax.imshow(video_reg[frame], origin='lower')
        ax.plot(*fixed_points[0], 'x', color='red')
        ax.plot(*fixed_points[1], 'x', color='blue')
        ax.set_title('Registered Image')
        ax.grid(True)

        # Second subplot: example placeholder (e.g., zoomed-in, or empty)
        ax2 = axes[1]
        ax2.imshow(head_reg[frame], origin='lower')  # You can change to something else
        ax2.plot(*fixed_points_after_reg[frame][0],  'x', color='red')
        ax2.plot(*fixed_points_after_reg[frame][1],  'x', color='blue')
        ax2.set_title('Debug View')
        ax2.grid(True)
        
        # Second subplot: example placeholder (e.g., zoomed-in, or empty)
        ax3 = axes[2]
        ax3.imshow(myVideo[:,:,frame],origin='lower')  # You can change to something else
        ax3.plot( *points_sy0[frame][0], 'x', color='red')
        ax3.plot( *points_sy0[frame][1], 'x', color='blue')
        ax3.plot(spline_sy0[frame][:, 0],spline_sy0[frame][:, 1],'-')
        ax3.set_title('Debug View Sy0')
        ax3.grid(True)

        plt.tight_layout()
        plt.show()

    interact(slider_movie, frame=IntSlider(value=11526, min=0, max=len_rec - 1, step=1))

In [ ]:
#%matplotlib widget
if plot_debug:
    def slider_movie(frame):
        plt.close()
        plt.figure(figsize=(5,2),dpi=250)
        plt.imshow(myVideo[:,:,frame],origin='lower')
        
        plt.plot(spline_sy0[frame][:, 0],spline_sy0[frame][:, 1],'r-')
        plt.plot(com_sy0[frame][0],com_sy0[frame][1],'x')
        plt.xlim(0,250)
        plt.ylim(0,250)
        #plt.title('final tracking')
        plt.title(reg_angle[frame])
        plt.plot((10,10+5//mm_per_pix),(10,10),'-',color='white', ms=5)
        plt.grid()
        plt.show()
    interact(slider_movie,frame=IntSlider(value=11085,min=0,max=len_rec-1,step=1))

# Reconstruction traj

In [ ]:
print(f'--Upload data relative to {run_n}')
for i,file in enumerate(os.listdir(path_experiment_runs)):
    if run_n in file:
        print(file)
        h5_experiment_run=os.path.join(path_experiment_runs,file)
        hdf5_file = h5py.File(h5_experiment_run, "r") 
        #h5tree_view(hdf5_file)
        cameratimestamp=hdf5_file['data/cameratimestamp'][:]
        cameratimes=hdf5_file['data/cameratimes'][:]
        date_f_run=hdf5_file['data/date_f_run'][:] 
        date_i_run=hdf5_file['data/date_i_run'][:] 
        frame_fish_yolo=hdf5_file['data/frame_fish_yolo'][:] 
        motorpositions=hdf5_file['data/motorpositions'][:] 
        motortimes=hdf5_file['data/motortimes'][:] 
        pos_fish_yolo=hdf5_file['data/pos_fish_yolo'][:] 
        trackingtimes=hdf5_file['data/trackingtimes'][:] 
        mm_per_pix=hdf5_file['exp/mm_per_pix'][:] 
      
tvals=cameratimes
print('--Process position: from pix to mm')
[com_pix,com_mm] = convert_pix2mm(r_com_sy0,xROIcenter,yROIcenter,mm_per_pix,tvals)
print('--Process spline: from pix to mm')
[spline_pix,spline_mm] = convert_pix2mm(r_spline_sy0,xROIcenter,yROIcenter,mm_per_pix,tvals)

cameratimes = cameratimes[:len(com_mm)]
tvals=cameratimes

print('--Process motor position and time')
motortimes_shift=motortimes-0.045

f_x_motor=interp1d(motortimes_shift,motorpositions[:,0],fill_value='extrapolate')
f_y_motor=interp1d(motortimes_shift,motorpositions[:,1],fill_value='extrapolate')

f_x_centroid_off=interp1d(cameratimes,com_mm[:,0],fill_value='extrapolate') 
f_y_centroid_off=interp1d(cameratimes,com_mm[:,1],fill_value='extrapolate') 

print('--Define the reconstructed trajectory')
X_off=f_x_motor(tvals)+f_x_centroid_off(tvals)
Y_off=f_y_motor(tvals)+f_y_centroid_off(tvals)

print('--Define the reconstructed spline')
aux_x_motor=f_x_motor(tvals)
aux_y_motor=f_y_motor(tvals)

X_spline=aux_x_motor[:, np.newaxis]+spline_mm[:,:,0]
Y_spline=aux_y_motor[:, np.newaxis]+spline_mm[:,:,1]

len_run=len(tvals)
print(f'**len_run={len_run}')

### check reconstruction

In [ ]:
#%matplotlib widget
plt.figure()
plt.plot(tvals,f_x_motor(tvals),'k-',alpha=1,label='motor pos')
plt.plot(tvals,X_off,'r-',ms=1,alpha=1,label='REC')
plt.plot(tvals,f_x_centroid_off(tvals)+120,'m-',alpha=1,label='fish pos')
plt.xlabel('(mm)')
plt.ylabel('(mm)')
plt.title('along x')
plt.legend()
#plt.ylim(95,110), plt.xlim(47,56)
plt.show()

In [ ]:
plt.figure()
plt.plot(tvals,f_y_motor(tvals),'k-',alpha=1,label='motor pos')
plt.plot(tvals,Y_off,'r-',ms=1,alpha=1,label='rec')
plt.plot(tvals,f_y_centroid_off(tvals)+312,'m-',alpha=1,label='fish pos')
#plt.ylim(125,160), plt.xlim(60,68)
plt.xlabel('(mm)')
plt.ylabel('(mm)')
plt.title('along y')
plt.legend()

plt.show()

# Re-sampling

In [ ]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
fontprops = fm.FontProperties(size=12)

fig, ax = plt.subplots()
plt.plot(motorpositions[:,0],motorpositions[:,1],'k-',alpha=0.1,label='motor position')
plt.plot(X_off,Y_off,'k.',alpha=0.01,label='rec.traj.')
plt.plot(X_off[0],Y_off[0],'s',label='start.traj.')
#plt.plot(X_ds,Y_ds,'y.-',alpha=1,label='ds.rec.traj')
#plt.scatter(X_ds[1:], Y_ds[1:], s=np.diff(t_ds)*100, c='y',alpha=1)
plt.gca().axis('equal')

#scalebar = AnchoredSizeBar(ax.transData, 4,'4 mm','lower center', pad=0.1, color='black',frameon=False, size_vertical=1, fontproperties=fontprops)
#ax.add_artist(scalebar)

plt.xlabel('x (mm)')
plt.ylabel('y (mm)')
plt.title('motor')
plt.legend()
plt.show()

## 👁‍🗨Speed

In [ ]:
%matplotlib inline
v2d_off=np.sqrt( np.gradient(X_off,tvals)**2 + np.gradient(Y_off,tvals)**2 )
#v2d_off_sav=savgol_filter(v2d_off,25,3) #(X,251//15,3)

plt.figure()
plt.plot(tvals,v2d_off,label='v2d_x')
#plt.plot(t_ds[:-1],v2d_ds,label='v2d_ds')
plt.xlabel('time (s)')
plt.ylabel(r'$speed$ (mm\s)')
plt.show()

# Orient

In [ ]:
orient=np.array([arr for sublist in r_orient for arr in sublist])

#%matplotlib widget
plt.figure()
plt.plot(tvals,np.sqrt( np.gradient(X_off,tvals)**2+np.gradient(Y_off,tvals)**2),'r-',ms=1,alpha=1,label='speed')
plt.plot(tvals,orient+450,label='orient')
plt.legend()
plt.show()

# Spline

## y_L

In [ ]:
[spline_mm_zero, y_L]=spline_transformed(len_run, spline_mm, orient)

In [ ]:
%matplotlib inline
def slider_movie(frame):
    plt.close()
    fig = plt.subplots(figsize=(5,3), dpi=150)
    plt.plot(spline_mm_zero[frame][:,0],spline_mm_zero[frame][:,1],'ro') #pos
    plt.plot(spline_mm_zero[frame][-1,0],y_L[frame],'go') 
    plt.title(f'{(orient[frame]+180)% 360}')
    plt.grid()
    plt.ylim([-10,10])
    plt.xlim([-1,10])
    plt.xlabel('(mm)')
    plt.show()
interact(slider_movie,frame=IntSlider(value=8599,min=0,max=len_run-1,step=1));

## lenght spline

In [ ]:
spline_length_mm=[]
for frame in range(len_run):
    spline_length_mm.append(calculate_spline_length(spline_mm_zero[frame]))

In [ ]:
avg_spline_l_mm = np.nanmean(spline_length_mm)
std_spline_l_mm = np.nanstd(spline_length_mm)

plt.figure()
plt.hist(spline_length_mm, bins=50, edgecolor='black', histtype='step')

plt.axvline(avg_spline_l_mm, color='red', linestyle='dashed', linewidth=1, label='Mean')
plt.axvline(avg_spline_l_mm - std_spline_l_mm, color='blue', linestyle='dashed', linewidth=1, label='-1 Std Dev')
plt.axvline(avg_spline_l_mm + std_spline_l_mm, color='blue', linestyle='dashed', linewidth=1, label='+1 Std Dev')

plt.title('Histogram with fill=off and Mean/Std Dev Lines')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.xlim([0,20])
plt.xlabel('(mm)')
plt.legend()
plt.show()

## data for PCA

In [ ]:
[dangles_spline_points, dangles_spline_points_nans, indices_pca]=PCA_spline(len_run, spline_mm_zero)

In [ ]:
movie_l=myVideo.shape[-1]

%matplotlib inline
def slider_movie(frame):
    plt.close()
    plt.figure(figsize=(10,4),dpi=150)
    plot1 = plt.subplot2grid((2, 3), (0, 0))
    plot2 = plt.subplot2grid((2, 3), (1, 0))
    plot3 = plt.subplot2grid((2, 3), (0, 1), rowspan=2, colspan=2)
    #plot4 = plt.subplot2grid((2, 5), (0, 3), rowspan=2, colspan=2)

    #zoomed traj
    plot1.plot(X_off[np.max([0,frame-500]):np.min([movie_l,frame+500])], Y_off[np.max([0,frame-500]):np.min([movie_l,frame+500])],'k.',ms=5,alpha=0.02)
    plot1.plot(X_off[frame], Y_off[frame],'ro',alpha=1)
    plot1.axis('equal')
    plot1.set_ylabel('x (mm)')
    plot1.set_xlabel('y (mm)')
    plot1.get_xaxis().set_visible(False)
    plot1.get_yaxis().set_visible(False)
    
    #traj
    plot2.plot(X_off, Y_off,'k-',ms=5,alpha=0.6)
    plot2.plot(X_off[frame], Y_off[frame],'ro',alpha=1)
    #plot2.plot([xA_abs,xB_abs,xC_abs,xD_abs,xA_abs],[yA_abs,yB_abs,yC_abs,yC_abs,yA_abs], 'k-' )
    plot2.axis('equal')
    plot2.set_ylabel('x (mm)')
    plot2.set_xlabel('y (mm)')
    
    #fish
    plot3.imshow(myVideo[:,:,frame],origin='lower',cmap='gray') 
    #if not np.isnan(spline_sy0[frame]).any():
    plot3.plot(spline_sy0[frame][2:, 0], spline_sy0[frame][2:, 1], 'g.',ms=2,alpha=1)
    #plot3.plot(spline_sy0[frame][-1, 0], spline_sy0[frame][-1, 1], 'y.',alpha=1)
    plot3.plot(com_sy0[frame][0], com_sy0[frame][1], 'r.',alpha=1)
    #if control_track_tail[frame]:
        #plot3.text(xROIcenter-20, yROIcenter+100, 'track failed', color='yellow') 
    #plot3.plot((10,10+5//mm_per_pix),(10,10),'-',color='white', ms=5)
    #plot3.text(20, 13, '5 mm', color='white')
    plot3.get_xaxis().set_visible(False)
    plot3.get_yaxis().set_visible(False)
    

      
    plt.tight_layout()
    plt.show()
    
    
interact(slider_movie,frame=IntSlider(value=10,min=0,max=movie_l-1,step=1));

# Saving

In [ ]:
tmp_name=str(run_n)+'_'+'postprocessing.h5'
file_name=os.path.join(path_postprocessed_runs,tmp_name)
print(f'{file_name}')

In [ ]:
hdf5_file = h5py.File(file_name,"w")

#saving
fish_id = hdf5_file.create_group("fish_id")
fish_id.attrs["description"] = "Define the fish identity"
fish_id.create_dataset("age", data=[age])
fish_id.create_dataset("spline_length_mm", data=[spline_length_mm])#mm 
fish_id.create_dataset("avg_spline_l_mm", data=[avg_spline_l_mm])#mm
fish_id.create_dataset("std_spline_l_mm", data=[std_spline_l_mm])#mm

exp = hdf5_file.create_group("exp")
exp.attrs["description"] = "Parameters of the experiment"
exp.create_dataset("fps", data=[fps])#hz
exp.create_dataset("mm_per_pix", data=mm_per_pix)
exp.create_dataset("start_exp", data=date_in_arena)
exp.create_dataset("start_run", data=date_i_run)
exp.create_dataset("end_run", data=date_f_run)

data = hdf5_file.create_group("data")
data.attrs["description"] = "Raw data coming from the experiment"
data.create_dataset("cameratimes", data=np.array(cameratimes))
data.create_dataset("cameratimestamp", data=np.array(cameratimestamp))
data.create_dataset("centroid_tot", data=np.array(pos_fish_yolo))
data.create_dataset("motorpositions", data=np.array(motorpositions))
data.create_dataset("motortimes", data=np.array(motortimes))
data.create_dataset("trackingtimes", data=np.array(trackingtimes))

post_process = hdf5_file.create_group("post-processing")
post_process.attrs["description"] = "Data after post processing the video"
#fish position in camera FOV (Sleap+registration)
post_process.create_dataset("com_pix", data=np.array(com_pix)) #pix #com: camera sys.
post_process.create_dataset("com_mm", data=np.array(com_mm)) #mm #com: camera sys.
#fish spline in camera FOV (Sleap+registration)
post_process.create_dataset("n_segments", data=[n_segments]) #spline tracking parameter
post_process.create_dataset("window_radius", data=[window_radius]) #spline tracking parameter
post_process.create_dataset("spline_pix", data=np.array(com_pix)) #pix #spline: camera sys.
post_process.create_dataset("spline_mm", data=np.array(com_mm)) #mm #spline: camera sys.
#fish position in LAB ref.
post_process.create_dataset("X", data=np.array(X_off)) #mm #com: LAB sys.
post_process.create_dataset("Y", data=np.array(Y_off)) #mm #com: LAB sys.
#fish spline in LAB ref.
post_process.create_dataset("X_spline", data=np.array(X_spline)) #mm #spline: LAB sys.
post_process.create_dataset("Y_spline", data=np.array(Y_spline)) #mm #spline: LAB sys.
#time
post_process.create_dataset("tvals", data=np.array(tvals)) #s
#speed
post_process.create_dataset("v2d_off", data=np.array(v2d_off)) #mm/s
#down sampling
#post_process.create_dataset("X_ds", data=X_ds)#mm
#post_process.create_dataset("Y_ds", data=Y_ds)#mm
#post_process.create_dataset("t_ds", data=t_ds)#s
#post_process.create_dataset("v2d_ds", data=v2d_ds)#mm/s
#post_process.create_dataset("dl", data=[dl])#mm
#orientation
post_process.create_dataset("orient", data=orient) #deg
#analysis spline, PCA
post_process.create_dataset("y_L", data=y_L) #mm #last pont of the tail
post_process.create_dataset("spline_mm_zero", data=spline_mm_zero) #translated in (0,0) and rotated of 'orient'

post_process.create_dataset("dangles_spline_points", data=dangles_spline_points)#from 'tail_points_zero'
post_process.create_dataset("dangles_spline_points_nans", data=dangles_spline_points_nans)
post_process.create_dataset("indices_pca", data=indices_pca)
hdf5_file.close()

In [ ]:
hdf5_file = h5py.File(file_name,"r")
h5tree_view(hdf5_file)
hdf5_file.close()

In [ ]:
# Quick check data

In [ ]:
hdf5_file = h5py.File(file_name,"r")
c_X=hdf5_file['post-processing/X'][:] 
c_Y=hdf5_file['post-processing/Y'][:] 

c_tvals=hdf5_file['post-processing/tvals'][:] 
c_v2d_off=hdf5_file['post-processing/v2d_off'][:] 
c_orient=hdf5_file['post-processing/orient'][:] 

c_spline_mm_zero=hdf5_file['post-processing/spline_mm_zero'][:] 

c_start_exp=hdf5_file['exp/start_exp'][:] 
c_start_run=hdf5_file['exp/start_run'][:] 
c_end_run=hdf5_file['exp/end_run'][:] 

hdf5_file.close()

print(f'{c_start_exp}-{c_start_run}-{c_end_run}')

In [ ]:
#%matplotlib widget
plt.figure()
plt.plot(c_X,c_Y, ',')
plt.axis('equal')
plt.title('check traj'),plt.xlabel('X(mm)'),plt.ylabel('Y(mm)')
plt.show()

In [ ]:
#%matplotlib widget
plt.figure()
plt.plot(c_tvals,c_orient, '-')
plt.title('check orient'),plt.xlabel('t(s)'),plt.ylabel('orient(deg)')
plt.show()

In [ ]:
%matplotlib inline
def slider_movie(frame):
    plt.close()
    fig = plt.subplots(figsize=(5,3), dpi=150)
    plt.plot(c_spline_mm_zero[frame][:,0],c_spline_mm_zero[frame][:,1],'ro') #pos
    plt.title('check spline')
    plt.grid()
    plt.ylim([-10,10])
    plt.xlim([-1,10])
    plt.xlabel('(mm)')
    plt.show()
interact(slider_movie,frame=IntSlider(value=8599,min=0,max=len_run-1,step=1));